# Multi-Agent Framework for Semantic Discovery and GraphRAG
## Interactive Demonstration

This notebook provides a step-by-step demonstration of the multi-agent pipeline.

### Pipeline

| Section | Purpose |
|---------|----------|
| 1 | Configuration |
| 2 | Environment check |
| 3 | Data loading |
| 4 | Builder Graph |
| 5 | Graph inspection |
| 6 | Query Graph |
| 7 | Cleanup |

### Prerequisites

Neo4j running on port 7687. LM Studio running on port 1234 with a model loaded.  configured with  and .


---
## Section 1 — Configuration

All settings are defined here. Modify this cell before running the rest of the notebook.


In [ ]:
from pathlib import Path

# ── Data paths ─────────────────────────────────────────────────────────────────
DATA_DIR = (Path.cwd().parent / "tests" / "fixtures").resolve()

# Smoke fixtures (fast): minimal dataset
DOC_FILES = [
    "smoke/business_glossary_smoke.txt",
    "smoke/data_dictionary_smoke.txt",
]
DDL_FILES = [
    "smoke/smoke_schema.sql",
]

# Full fixtures (complete run):
# DOC_FILES = ["sample_docs/business_glossary.txt", "sample_docs/data_dictionary.txt"]
# DDL_FILES  = ["sample_ddl/simple_schema.sql"]

# ── LLM model selection ────────────────────────────────────────────────────────
# Reasoning LLM: used for mapping, critic, ER judge, Cypher gen/healing
LLM_MODEL_REASONING = "openai/gpt-oss-120b"

# Extraction SLM: used for triplet extraction from PDF/text chunks
LLM_MODEL_EXTRACTION = "local-model"   # LM Studio local

# API keys (only needed for the providers you actually use above)
OPENROUTER_API_KEY = ""     # Required when using OpenRouter models (provider/model format)
# OPENAI_API_KEY   = ""     # Required for OpenAI direct ("gpt-*", "o1-*")
# ANTHROPIC_API_KEY = ""    # Required for Anthropic direct ("claude-*")

# Token budgets
LLM_MAX_TOKENS_REASONING  = 16384   # Caps thinking+output on reasoning LLM (avoids empty responses)
LLM_MAX_TOKENS_EXTRACTION = 16384   # Caps JSON output on extraction SLM

TEMPERATURE_EXTRACTION = 0.0
TEMPERATURE_REASONING  = 0.0
TEMPERATURE_GENERATION = 0.3

# LM Studio endpoint (ignored when EXTRACTION model is not a local model)
LMSTUDIO_BASE_URL = "http://localhost:1234/v1"

# ── Neo4j ──────────────────────────────────────────────────────────────────────
NEO4J_URI      = "bolt://localhost:7687"
NEO4J_USER     = "neo4j"
NEO4J_PASSWORD = "test_password"

# ── Entity resolution ──────────────────────────────────────────────────────────
ER_BLOCKING_TOP_K       = 10
ER_SIMILARITY_THRESHOLD = 0.75

# ── Retrieval ──────────────────────────────────────────────────────────────────
RETRIEVAL_MODE         = "hybrid"
RETRIEVAL_VECTOR_TOP_K = 20
RETRIEVAL_BM25_TOP_K   = 10
RETRIEVAL_GRAPH_DEPTH  = 2
RERANKER_TOP_K         = 5

# ── Thresholds ─────────────────────────────────────────────────────────────────
CONFIDENCE_THRESHOLD        = 0.90
MAX_REFLECTION_ATTEMPTS     = 3
MAX_CYPHER_HEALING_ATTEMPTS = 3
MAX_HALLUCINATION_RETRIES   = 3

# ── Ablation flags ─────────────────────────────────────────────────────────────
ENABLE_SCHEMA_ENRICHMENT    = True
ENABLE_CYPHER_HEALING       = True
ENABLE_CRITIC_VALIDATION    = True
ENABLE_RERANKER             = True
ENABLE_HALLUCINATION_GRADER = True

# ── Logging ────────────────────────────────────────────────────────────────────
LOG_LEVEL = "WARNING"   # WARNING (minimal) | INFO (verbose) | DEBUG (full trace)

# ── Summary ────────────────────────────────────────────────────────────────────
from src.config.llm_factory import detect_provider
_r_provider = detect_provider(LLM_MODEL_REASONING)
_e_provider = detect_provider(LLM_MODEL_EXTRACTION)
print("Configuration loaded.")
print(f"  Reasoning  : {LLM_MODEL_REASONING!r}  →  provider={_r_provider}  max_tokens={LLM_MAX_TOKENS_REASONING}")
print(f"  Extraction : {LLM_MODEL_EXTRACTION!r}  →  provider={_e_provider}  max_tokens={LLM_MAX_TOKENS_EXTRACTION}")
print(f"  Documents  : {len(DOC_FILES)} file(s)")
print(f"  DDL files  : {len(DDL_FILES)} file(s)")
print(f"  Neo4j      : {NEO4J_URI}")


Configuration loaded.
  Documents : 2 file(s)
  DDL files : 1 file(s)
  LM Studio : http://localhost:1234/v1  model=qwen/qwen3.5-122b-a10b
  Neo4j     : bolt://localhost:7687


---
## Section 2 — Environment Check

Verify that Neo4j is reachable and the LM Studio server is responding.


In [2]:
import sys, os
import requests

repo_root = Path.cwd().parent
sys.path.insert(0, str(repo_root))

# Push all config into environment so Pydantic Settings picks them up
os.environ["NEO4J_URI"]                    = NEO4J_URI
os.environ["NEO4J_USER"]                   = NEO4J_USER
os.environ["NEO4J_PASSWORD"]               = NEO4J_PASSWORD
os.environ["LMSTUDIO_BASE_URL"]            = LMSTUDIO_BASE_URL
os.environ["LLM_MODEL_REASONING"]          = LLM_MODEL_REASONING
os.environ["LLM_MODEL_EXTRACTION"]         = LLM_MODEL_EXTRACTION
os.environ["LLM_MAX_TOKENS_EXTRACTION"]    = str(LLM_MAX_TOKENS_EXTRACTION)
os.environ["LLM_MAX_TOKENS_REASONING"]     = str(LLM_MAX_TOKENS_REASONING)
os.environ["LOG_LEVEL"]                    = LOG_LEVEL
if OPENROUTER_API_KEY:
    os.environ["OPENROUTER_API_KEY"]       = OPENROUTER_API_KEY

# Clear cached LLM instances so they re-read the updated env vars
from src.config.llm_factory import reconfigure_from_env
reconfigure_from_env()

from src.graph.neo4j_client import Neo4jClient

# Neo4j check
neo4j_ok = False
try:
    with Neo4jClient() as client:
        client.execute_cypher("RETURN 1")
    neo4j_ok = True
except Exception as exc:
    print(f"[FAIL] Neo4j: {exc}")

# LM Studio check (only when extraction model is local)
from src.config.llm_factory import detect_provider
lmstudio_ok = detect_provider(LLM_MODEL_EXTRACTION) == "lmstudio"  # skip if not local
model_ids = []
if lmstudio_ok:
    lmstudio_ok = False
    try:
        resp = requests.get(f"{LMSTUDIO_BASE_URL}/models", timeout=5)
        if resp.status_code == 200:
            model_ids = [m["id"] for m in resp.json().get("data", [])]
            lmstudio_ok = True
    except Exception as exc:
        print(f"[FAIL] LM Studio: {exc}")

print("Environment Check")
print("-" * 40)
print(f"Neo4j      : {'OK' if neo4j_ok else 'FAIL'}  ({NEO4J_URI})")
if detect_provider(LLM_MODEL_EXTRACTION) == "lmstudio":
    print(f"LM Studio  : {'OK' if lmstudio_ok else 'FAIL'}  ({LMSTUDIO_BASE_URL})")
    if model_ids:
        print(f"  Loaded models: {model_ids}")
else:
    print(f"LM Studio  : skipped (extraction uses {detect_provider(LLM_MODEL_EXTRACTION)})")
print("-" * 40)
if not neo4j_ok:
    print("Fix the issues above before proceeding.")
else:
    print("All required services are reachable.")


/home/marcantoniolopez/Documenti/github/thesis/.venv/lib/python3.12/site-packages/pydantic_settings/sources/providers/secrets.py:67: UserWarning: directory "/run/secrets" does not exist
  warnings.warn(f'directory "{path}" does not exist')


Environment Check
----------------------------------------
Neo4j     : OK  (bolt://localhost:7687)
LM Studio : OK  (http://localhost:1234/v1)
  Loaded models: ['qwen/qwen3.5-35b-a3b', 'text-embedding-nomic-embed-text-v1.5']
----------------------------------------
All services are reachable.


---
## Section 3 — Data Loading

Load and chunk source documents, then parse DDL files.


In [3]:
from src.ingestion.pdf_loader import load_pdf, chunk_documents
from src.ingestion.ddl_parser import parse_ddl

doc_paths = [DATA_DIR / f for f in DOC_FILES]
ddl_paths = [DATA_DIR / f for f in DDL_FILES]

all_chunks = []
for path in doc_paths:
    docs   = load_pdf(path)
    chunks = chunk_documents(docs)
    all_chunks.extend(chunks)

all_tables = []
for path in ddl_paths:
    ddl = path.read_text()
    all_tables.extend(parse_ddl(ddl))

print(f"Documents loaded : {len(doc_paths)} file(s)  ->  {len(all_chunks)} chunk(s)")
print(f"DDL files loaded : {len(ddl_paths)} file(s)  ->  {len(all_tables)} table(s)")
if all_chunks:
    print()
    print("First chunk preview:")
    print("-" * 40)
    print(all_chunks[0].text[:300])


/home/marcantoniolopez/Documenti/github/thesis/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Documents loaded : 2 file(s)  ->  2 chunk(s)
DDL files loaded : 1 file(s)  ->  2 table(s)

First chunk preview:
----------------------------------------
BUSINESS GLOSSARY — SMOKE TEST

Customer
--------
A Customer is an individual or organisation that has placed at least one
confirmed order on the platform. A Customer has a unique identifier,
a full name, an email address, and a region code.

Product
-------
A Produc


---
## Section 4 — Builder Graph

Orchestrates the full Knowledge Graph construction pipeline.

| Step | Component | Description |
|------|-----------|-------------|
| 1 | Triplet Extractor | Extract (subject, predicate, object) from text |
| 2 | Entity Resolver | Deduplicate entities: embedding blocking + LLM judge |
| 3 | DDL Parser | Parse SQL into typed table definitions |
| 4 | Schema Enricher | Expand abbreviated column names via LLM |
| 5 | RAG Mapper | Map each physical table to a business concept |
| 6 | Validator | Actor-Critic validation of mapping proposals |
| 7 | Cypher Generator | Generate MERGE statements for Neo4j |
| 8 | Cypher Healer | Auto-correct syntax errors via self-reflection |
| 9 | Graph Writer | Execute Cypher against Neo4j |

Self-reflection loops (via ) are active at steps 1, 2, 5, and 8.

### Node diagram




In [ ]:
import time
import logging
from src.graph.builder_graph import run_builder

print("Running Builder Graph ...")

# Reconfigure all handlers to INFO for builder graph execution
for _handler in logging.getLogger().handlers:
    _handler.setLevel(logging.INFO)
logging.getLogger().setLevel(logging.INFO)

start = time.time()

final_state = run_builder(
    raw_documents=doc_paths,
    ddl_paths=ddl_paths,
    production=False,
    clear_graph=True,  # wipe stale data from prior demo runs
)

elapsed = time.time() - start

triplets  = final_state.get("triplets", [])
entities  = final_state.get("entities", [])
tables    = final_state.get("tables", [])
enriched  = final_state.get("enriched_tables", [])
completed = final_state.get("completed_tables", [])
failed    = final_state.get("cypher_failed", False)

print()
print("Builder Graph Results")
print("-" * 40)
print(f"Elapsed             : {elapsed:.1f} s")
print(f"Triplets extracted  : {len(triplets)}")
print(f"Entities resolved   : {len(entities)}")
print(f"Tables parsed       : {len(tables)}")
print(f"Tables enriched     : {len(enriched)}")
print(f"Tables completed    : {len(completed)}")
print(f"Cypher failures     : {failed}")
if completed:
    print()
    print("Completed tables:")
    for t in completed:
        print(f"  - {t}")


{"ts": "2026-03-13T16:51:23", "logger": "src.ingestion.pdf_loader", "level": "INFO", "message": "Loaded text file 'business_glossary_smoke.txt'"}
{"ts": "2026-03-13T16:51:23", "logger": "src.ingestion.pdf_loader", "level": "INFO", "message": "Chunked 1 documents into 1 chunks (chunk_size=512, overlap=64)"}
{"ts": "2026-03-13T16:51:23", "logger": "src.ingestion.pdf_loader", "level": "INFO", "message": "Loaded text file 'data_dictionary_smoke.txt'"}
{"ts": "2026-03-13T16:51:23", "logger": "src.ingestion.pdf_loader", "level": "INFO", "message": "Chunked 1 documents into 1 chunks (chunk_size=512, overlap=64)"}
{"ts": "2026-03-13T16:51:23", "logger": "src.graph.builder_graph", "level": "INFO", "message": "Graph cleared before builder run."}
{"ts": "2026-03-13T16:51:23", "logger": "src.graph.neo4j_client", "level": "INFO", "message": "Running schema setup (4 statements)..."}
{"ts": "2026-03-13T16:51:23", "logger": "src.graph.neo4j_client", "level": "INFO", "message": "Schema setup complete."

Running Builder Graph ...


/home/marcantoniolopez/Documenti/github/thesis/.venv/lib/python3.12/site-packages/langgraph/_internal/_runnable.py:400: UserWarning: Parameters {'extra_body'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  ret = self.func(*args, **kwargs)
{"ts": "2026-03-13T16:53:06", "logger": "httpx", "level": "INFO", "message": "HTTP Request: POST http://localhost:1234/v1/chat/completions \"HTTP/1.1 200 OK\""}
{"ts": "2026-03-13T16:53:06", "logger": "llm.extraction", "level": "INFO", "message": "llm.extraction call completed | attempt=1 | latency_ms=102469.8 | input_tokens=487 | output_tokens=972 | total_tokens=1459"}
{"ts": "2026-03-13T16:53:06", "logger": "src.extraction.triplet_extractor", "level": "INFO", "message": "Chunk 0 \u2192 13 triplets extracted (source=business_glossary_smoke.txt, page=1)"}
{"ts": "2026-03-13T16:54:46", "logger": "httpx", "level": "INFO", "message": "HTTP Request: POST http://localhost:1234/v1/chat/completions \"HTTP/1

---
## Section 5 — Graph Inspection

Query the Knowledge Graph using Cypher to verify the built ontology.


In [ ]:
from src.graph.neo4j_client import Neo4jClient

with Neo4jClient() as client:
    node_rows = client.execute_cypher(
        "MATCH (n) RETURN labels(n) AS labels, count(n) AS count ORDER BY count DESC"
    )
    print("Node counts")
    print("-" * 40)
    for r in node_rows:
        print(f"  {r['labels']}: {r['count']}")

    concept_rows = client.execute_cypher(
        "MATCH (c:BusinessConcept) RETURN c.name AS name LIMIT 20"
    )
    print()
    print("Business Concepts")
    print("-" * 40)
    for r in concept_rows:
        print(f"  {r['name']}")

    mapping_rows = client.execute_cypher("""
        MATCH (c:BusinessConcept)-[r:MAPPED_TO]->(t:PhysicalTable)
        RETURN c.name AS concept, t.table_name AS table,
               round(r.confidence * 100) AS pct
        ORDER BY r.confidence DESC
    """)
    print()
    print("Concept -> Table Mappings")
    print("-" * 40)
    for r in mapping_rows:
        print(f"  {r['concept']} -> {r['table']}  ({r['pct']}%)")


Node counts
----------------------------------------

Business Concepts
----------------------------------------

Concept -> Table Mappings
----------------------------------------


### Graph Visualization
Interactive pyvis rendering of the Knowledge Graph (drag, zoom, hover for details).

In [ ]:
from pyvis.network import Network
from IPython.display import IFrame, display
from src.graph.neo4j_client import Neo4jClient

net = Network(
    height="520px", width="100%",
    bgcolor="#1a1a2e", font_color="white",
    directed=True,
)
net.set_options("""
{
  "physics": {"barnesHut": {"gravitationalConstant": -8000, "springLength": 160}},
  "nodes": {"font": {"size": 14}},
  "edges": {"arrows": {"to": {"enabled": true}}}
}
""")

# ── colour palette per label ───────────────────────────────────────────
_COLORS = {
    "BusinessConcept": "#e94560",
    "PhysicalTable":   "#0f3460",
    "default":         "#533483",
}

with Neo4jClient() as client:
    # Nodes
    rows = client.execute_cypher(
        "MATCH (n) RETURN id(n) AS nid, labels(n) AS lbls, "
        "COALESCE(n.name, n.table_name, 'node') AS label, "
        "COALESCE(n.definition, n.ddl_source, '') AS tooltip"
    )
    for r in rows:
        lbl   = r["lbls"][0] if r["lbls"] else "default"
        color = _COLORS.get(lbl, _COLORS["default"])
        net.add_node(
            r["nid"], label=r["label"],
            color=color, title=f"[{lbl}]\n{r['tooltip'][:120]}",
            size=28 if lbl == "BusinessConcept" else 20,
        )

    # Edges
    rels = client.execute_cypher(
        "MATCH (a)-[r]->(b) RETURN id(a) AS src, id(b) AS tgt, "
        "type(r) AS rel_type, "
        "COALESCE(r.confidence, r.column, '') AS prop"
    )
    for r in rels:
        prop  = f" ({r['prop']:.0%})" if isinstance(r["prop"], float) else (
                f" [{r['prop']}]" if r["prop"] else ""
        )
        net.add_edge(
            r["src"], r["tgt"],
            label=r["rel_type"] + prop,
            color="#a8a8b3",
        )

out = "../notebooks/kg_preview.html"
net.write_html(out, notebook=False)
display(IFrame(src="kg_preview.html", width="100%", height="540px"))
print(f"Saved to {out}")


{"ts": "2026-03-13T16:42:56", "logger": "neo4j.notifications", "level": "WARNING", "message": "Received notification from DBMS server: <GqlStatusObject gql_status='01N02', status_description='warn: feature deprecated without replacement. id is deprecated and will be removed without a replacement.', position=<SummaryInputPosition line=1, column=18, offset=17>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 17, 'line': 1, 'column': 18}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: \"MATCH (n) RETURN id(n) AS nid, labels(n) AS lbls, COALESCE(n.name, n.table_name, 'node') AS label, COALESCE(n.definition, n.ddl_source, '') AS tooltip\""}
{"ts": "2026-03-13T16:42:56", "logger": "neo4j.notifications", "level": "WARNING", "message":

Saved to ../notebooks/kg_preview.html


---
## Section 6 — Query Graph

Answer natural-language questions using the Knowledge Graph.

| Step | Component | Description |
|------|-----------|-------------|
| 1 | Hybrid Retriever | Dense + BM25 + Graph, fused via Reciprocal Rank Fusion |
| 2 | Cross-Encoder Reranker | Re-scores chunks for precision |
| 3 | Answer Generator | LLM generates grounded answer |
| 4 | Hallucination Grader | Self-RAG audit; retries with critique if needed |

### Node diagram




In [ ]:
from src.generation.query_graph import run_query

user_query = "Which table stores customer data and what columns does it have?"

print(f"Query: {user_query}")
print()

result = run_query(user_query)

print("Answer")
print("-" * 40)
print(result.get("final_answer", "No answer generated."))


{"ts": "2026-03-13T16:42:56", "logger": "src.graph.neo4j_client", "level": "INFO", "message": "Running schema setup (4 statements)..."}
{"ts": "2026-03-13T16:42:56", "logger": "src.graph.neo4j_client", "level": "INFO", "message": "Schema setup complete."}


Query: Which table stores customer data and what columns does it have?



{"ts": "2026-03-13T16:42:57", "logger": "httpx", "level": "INFO", "message": "HTTP Request: POST https://openrouter.ai/api/v1/chat/completions \"HTTP/1.1 400 Bad Request\""}


BadRequestError: Error code: 400 - {'error': {'message': 'Reasoning is mandatory for this endpoint and cannot be disabled.', 'code': 400, 'metadata': {'provider_name': None}}}

In [ ]:
test_questions = [
    "What entities exist in the business domain?",
    "Which table stores customer information?",
    "How are customers and orders related?",
]

print("Batch Query Results")
print("=" * 60)
for i, question in enumerate(test_questions, 1):
    result = run_query(question)
    answer = result.get("final_answer", "")
    if len(answer) > 300:
        answer = answer[:300] + " [truncated]"
    print(f"[{i}/{len(test_questions)}] {question}")
    print(f"    {answer}")
    print()

{"ts": "2026-03-13T16:12:42", "logger": "src.graph.neo4j_client", "level": "INFO", "message": "Running schema setup (4 statements)..."}
{"ts": "2026-03-13T16:12:42", "logger": "src.graph.neo4j_client", "level": "INFO", "message": "Schema setup complete."}


Batch Query Results


{"ts": "2026-03-13T16:12:42", "logger": "src.retrieval.reranker", "level": "WARNING", "message": "Reranker scoring failed (CUDA out of memory. Tried to allocate 16.00 MiB. GPU 0 has a total capacity of 7.53 GiB of which 15.69 MiB is free. Process 38385 has 128.00 MiB memory in use. Process 190601 has 6.24 GiB memory in use. Including non-PyTorch memory, this process has 1.12 GiB memory in use. Of the allocated memory 1010.64 MiB is allocated by PyTorch, and 5.36 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)) \u2014 returning chunks unranked."}
{"ts": "2026-03-13T16:12:49", "logger": "httpx", "level": "INFO", "message": "HTTP Request: POST http://localhost:1234/v1/chat/completions \"HTTP/1.1 200 OK\""}
{"ts": "2026-03-13T16:12:49", "logger": "llm.reasonin

[1/3] What entities exist in the business domain?
    Based on the retrieved context, the entities existing in the business domain are the **Customer Identity Record** and the **Sales Order Record**. The Customer Identity Record is described as containing core identity and contact attributes such as ID, name, email, and region, with a specific note tha [truncated]



{"ts": "2026-03-13T16:13:25", "logger": "src.retrieval.reranker", "level": "WARNING", "message": "Reranker scoring failed (CUDA out of memory. Tried to allocate 16.00 MiB. GPU 0 has a total capacity of 7.53 GiB of which 15.69 MiB is free. Process 38385 has 128.00 MiB memory in use. Process 190601 has 6.24 GiB memory in use. Including non-PyTorch memory, this process has 1.12 GiB memory in use. Of the allocated memory 1010.64 MiB is allocated by PyTorch, and 5.36 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)) \u2014 returning chunks unranked."}
{"ts": "2026-03-13T16:13:35", "logger": "httpx", "level": "INFO", "message": "HTTP Request: POST http://localhost:1234/v1/chat/completions \"HTTP/1.1 200 OK\""}
{"ts": "2026-03-13T16:13:35", "logger": "llm.reasonin

[2/3] Which table stores customer information?
    [Source: Web Search] `customer_id`: A surrogate primary key used to uniquely identify each customer in the table.
 `store_id`: A foreign key identifying the customer “home store.” Customers are not limited to renting only from this store, but this is the store at which they generally shop.
 `first_n [truncated]



{"ts": "2026-03-13T16:14:40", "logger": "src.retrieval.reranker", "level": "WARNING", "message": "Reranker scoring failed (CUDA out of memory. Tried to allocate 16.00 MiB. GPU 0 has a total capacity of 7.53 GiB of which 15.69 MiB is free. Process 38385 has 128.00 MiB memory in use. Process 190601 has 6.24 GiB memory in use. Including non-PyTorch memory, this process has 1.12 GiB memory in use. Of the allocated memory 1010.64 MiB is allocated by PyTorch, and 5.36 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)) \u2014 returning chunks unranked."}
{"ts": "2026-03-13T16:14:46", "logger": "httpx", "level": "INFO", "message": "HTTP Request: POST http://localhost:1234/v1/chat/completions \"HTTP/1.1 200 OK\""}
{"ts": "2026-03-13T16:14:46", "logger": "llm.reasonin

[3/3] How are customers and orders related?
    The provided context does not contain information describing how customers and orders are related, nor does it specify any relationships or linkage between the CUST_MASTER table and the SALES_ORD table. While the text identifies SALES_ORD as storing individual sales order records and describes a tab [truncated]



---
## Section 7 — Cleanup

Delete all nodes from the Knowledge Graph to reset the environment.


In [ ]:
# WARNING: this permanently deletes all graph data.
# Uncomment and run to reset.

# from src.graph.neo4j_client import Neo4jClient
# with Neo4jClient() as client:
#     client.execute_cypher("MATCH (n) DETACH DELETE n")
# print("Knowledge Graph cleared.")

print("Cleanup cell ready. Uncomment the code above to delete the graph.")


Cleanup cell ready. Uncomment the code above to delete the graph.


---
## Summary

| Phase | Input | Output |
|-------|-------|--------|
| Builder Graph | Documents + DDL | Knowledge Graph in Neo4j |
| Query Graph | Natural-language question | Grounded answer |

**Next steps:**
- Replace smoke fixtures with your own documents in Section 1
- Toggle ablation flags to compare pipeline variants
- Explore the graph at http://localhost:7474

**Documentation:** 


In [ ]:
#"""
from src.graph.neo4j_client import Neo4jClient

with Neo4jClient() as client:
    client.execute_cypher("MATCH (n) DETACH DELETE n")
print("Knowledge Graph cleared.")
#"""

Knowledge Graph cleared.


/home/marcantoniolopez/Documenti/github/thesis/.venv/lib/python3.12/site-packages/pydantic_settings/sources/providers/secrets.py:67: UserWarning: directory "/run/secrets" does not exist
  warnings.warn(f'directory "{path}" does not exist')
